# RL — CartPole Agent (real gymnasium run)

Trains a simple policy-gradient agent on CartPole-v1 using gymnasium. Downloads nothing but requires gymnasium.

_Last rebuild: **2026-02-16 03:25:33**_

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


In [2]:
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n

class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, act_dim)
        )
    def forward(self, x):
        return self.net(x)

pi = Policy().to(device)
opt = optim.Adam(pi.parameters(), lr=1e-2)

def run_episode(seed=None):
    obs, _ = env.reset(seed=seed)
    logps, rews = [], []
    done = False
    while not done:
        x = torch.tensor(obs, dtype=torch.float32, device=device)
        logits = pi(x)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        logps.append(dist.log_prob(a))
        obs, r, term, trunc, _ = env.step(int(a))
        done = term or trunc
        rews.append(r)
    return logps, rews

def returns(rews, gamma=0.99):
    G = 0.0
    out = []
    for r in reversed(rews):
        G = r + gamma * G
        out.append(G)
    out = list(reversed(out))
    out = torch.tensor(out, dtype=torch.float32, device=device)
    return (out - out.mean()) / (out.std() + 1e-8)

for ep in range(80):
    logps, rews = run_episode(seed=42)
    R = returns(rews)
    loss = -(torch.stack(logps) * R).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep+1) % 20 == 0:
        print('episode', ep+1, 'reward', sum(rews))

env.close()


episode 20 reward 113.0
episode 40 reward 152.0
episode 60 reward 215.0
episode 80 reward 413.0


In [3]:
print('DONE 2026-02-16 03:25:33')

DONE 2026-02-16 03:25:33
